In [4]:
import pandas as pd
from geopy.geocoders import GoogleV3
from time import sleep
import os
from dotenv import load_dotenv
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import folium
from folium import GeoJson


In [5]:

# Load API key from .env
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
if not API_KEY:
    raise ValueError("Google API key not found.")

# Initialize geocoder
geolocator = GoogleV3(api_key=API_KEY)

# Set project-relative paths
base_dir = Path.cwd()
input_path = base_dir / "data" / "complete_sets" / "arkansas_colleges_starter.csv"
output_path = base_dir / "ar_outputs" / "arkansas_colleges_geocoded.csv"

# Load CSV
df = pd.read_csv(input_path)

# Geocoding function
def geocode_location(row):
    try:
        query = f"{row['Institution']}, {row['City']}, Arkansas"
        location = geolocator.geocode(query, timeout=10)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except Exception as e:
        print(f"Error geocoding {row['Institution']}: {e}")
    return pd.Series([None, None])

# Apply and save
print("Starting geocoding...")
df[['Latitude', 'Longitude']] = df.apply(geocode_location, axis=1)
df.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")
df.head()


Starting geocoding...
✅ Saved: c:\Users\balla\Projects\Ballard-Capstone-proj\ar_outputs\arkansas_colleges_geocoded.csv


,Institution,City,Type,Latitude,Longitude
0,University of Arkansas,Fayetteville,4-year,36.068689,-94.174847
1,Arkansas State University,Jonesboro,4-year,35.843086,-90.674859
2,University of Central Arkansas,Conway,4-year,35.078094,-92.457892
3,Southern Arkansas University,Magnolia,4-year,33.293097,-93.232942
4,University of Arkansas at Pine Bluff,Pine Bluff,4-year,34.247369,-92.021742


In [6]:
# Load your geocoded CSV
college_df = pd.read_csv("ar_outputs/arkansas_colleges_geocoded.csv")

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(college_df['Longitude'], college_df['Latitude'])]
gdf_colleges = gpd.GeoDataFrame(college_df, geometry=geometry, crs="EPSG:4326")  # WGS84

# Project to a CRS that uses meters (EPSG:5070 is good for U.S.)
gdf_colleges_proj = gdf_colleges.to_crs(epsg=5070)

# Buffer each point by ~30 miles (48,280 meters)
gdf_colleges_proj['buffer_30mi'] = gdf_colleges_proj.geometry.buffer(48280)

# Convert buffer zones back to lat/lon for mapping
gdf_buffers = gdf_colleges_proj.set_geometry('buffer_30mi').to_crs(epsg=4326)


# Center on Arkansas
m = folium.Map(location=[34.75, -92.25], zoom_start=7)

# Add buffer zones
GeoJson(gdf_buffers[['Institution', 'buffer_30mi']], name="30-Mile Zones").add_to(m)

# Add markers for each college
for idx, row in gdf_colleges.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=row['Institution'],
        icon=folium.Icon(color='blue', icon='university', prefix='fa')
    ).add_to(m)

# Display map
m
